# Interactive Denoising Lab — PandaOmron + GRPO LoRA — **Multi-Checkpoint** (v2)

Same seed-sweep analysis as `interactive_denoising_panda_seed_305067_lora_jitter_v1.ipynb`
(its Cell 11, "Compare seeds"), but run for **many GRPO LoRA checkpoints in a single
notebook**, against one shared observation. Each checkpoint produces **one composite
figure** (large 3D EEF-trajectory panel + per-component traces + mean±std spread).

## How this works — one model load, one feature encode

The Eagle VLM backbone is **frozen and never touched by LoRA** — LoRA only wraps
`nn.Linear` modules inside the DiT action head. That makes a single-notebook sweep
straightforward:

1. **Load the 3B model once** (Cell 2).
2. **Inject LoRA once** at `rank=16, alpha=32` and **do NOT merge** (Cell 2.5).
   Merging folds the adapter into the base weights *irreversibly*; we keep it as an
   overlay so each checkpoint's weights can be swapped in for its run.
3. **Encode backbone features once per observation** (inside `analyze_all_checkpoints`).
   Because LoRA never touches the backbone, the same `features` are valid input for
   every checkpoint.
4. **For each checkpoint**: `load_lora_checkpoint(...)` overwrites the adapter's A/B
   matrices in place (fully replacing the previous checkpoint), then we run the
   100-seed sweep and draw **one composite figure per checkpoint**.

`load_lora_checkpoint` does a strict two-sided key + shape check, so a checkpoint
trained with a different rank/target-module set would hard-fail here rather than
silently degrade. All 10 checkpoints below share the GRPO defaults
(`rank=16, alpha=32`, `DEFAULT_LORA_TARGET_MODULES`); their `lora_weights.pt` files
are byte-identical in size, confirming the same adapter layout.

> ⚠️ The LoRA **scale** (`alpha / rank`) comes from the `apply_lora_to_dit` call,
> *not* from the checkpoint file. If any checkpoint was trained with a different
> `alpha`, set `LORA_ALPHA` to match it before loading — otherwise the adapter is
> applied at the wrong scale.

To analyze a different observation, just call
`analyze_all_checkpoints("/path/to/other.npz")` in a new cell — no reload needed.

In [ ]:
# Cell 1: Imports + config
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from scripts.denoising_lab.denoising_lab import DenoisingLab, TrajectoryVisualizer

MODEL_PATH = "nvidia/GR00T-N1.6-3B"
EMBODIMENT_TAG = "robocasa_panda_omron"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Observation .npz captured by interactive_rollout.py. Change this (or just call
# analyze_all_checkpoints("...") with another path) for any other observation.
OBS_PATH = "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step010.npz"

EEF_KEY = "end_effector_position"
N_SEEDS = 100              # seeds per checkpoint (matches v1 Cell 11)

# LoRA config — MUST match what the checkpoints were trained with.
# (GRPOConfig defaults; all 10 checkpoints below use these.)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

# Checkpoints to compare (relative to REPO_ROOT). Each dir holds lora_weights.pt.
CHECKPOINTS = [
    "grpo_data/overfit_step10_vanilla/iter_0002",
    "grpo_data/overfit_step10_jitter_balanced_lr1e5/iter_0009",
    "grpo_data/overfit_step10_jitter_balanced_lr1e5/iter_0004",
    "grpo_data/overfit_step10_balanced_lr3e6/iter_0006",
    "grpo_data/overfit_step10_balanced_lr3e6/iter_0004",
    "grpo_data/overfit_step10_jitter_balanced_lr3e5_from9/iter_0012",
    "grpo_data/overfit_step10_jitter_balanced_lr3e5_from9/iter_0011",
    "grpo_data/overfit_step10_jitter_balanced_lr3e5_from9/iter_0010",
    "grpo_data/overfit_step10_balanced/iter_0003",
    "grpo_data/overfit_step10_jitter_balanced_lam0.1/iter_0002",
]

print(f"Device: {DEVICE}")
print(f"{len(CHECKPOINTS)} checkpoints, {N_SEEDS} seeds each")

In [ ]:
# Cell 2: Load model (once)
lab = DenoisingLab(MODEL_PATH, EMBODIMENT_TAG, device=DEVICE)
print(f"Model loaded. Action horizon: {lab.action_horizon}, Action dim: {lab.action_dim}")
print(f"Inference timesteps: {lab.num_inference_timesteps}, Timestep buckets: {lab.num_timestep_buckets}")

In [ ]:
# Cell 2.5: Inject LoRA ONCE into the DiT — do NOT load a checkpoint yet, do NOT merge.
#
# We inject the adapter shell here and reuse it for every checkpoint: the loop below
# calls load_lora_checkpoint(...) to swap each checkpoint's A/B matrices into these
# same modules. We deliberately skip merge_lora_weights() — merging is irreversible
# and would bake one checkpoint into the base weights, making it impossible to cleanly
# load the next one. The cost is a small per-forward LoRA matmul (a few extra seconds
# per 100-seed sweep), which is fine here.
import sys as _sys
_grpo_dir = os.path.join(REPO_ROOT, "scripts", "grpo")
if _grpo_dir not in _sys.path:
    _sys.path.insert(0, _grpo_dir)

from lora_dit import apply_lora_to_dit, load_lora_checkpoint, print_trainable_params

apply_lora_to_dit(lab.model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)

# Pin the freshly-injected LoRA Linears to the DiT's device/dtype. Done ONCE:
# load_lora_checkpoint copies weights in place via load_state_dict, so loaded adapters
# inherit this device/dtype on every subsequent load (no re-.to() needed).
lab.model.action_head.model.to(device=lab.device, dtype=lab.dtype)
lab.model.eval()

print_trainable_params(lab.model)  # ~14.5M LoRA params, rest frozen

## Per-checkpoint seed sweep

`run_seed_sweep` reproduces v1 Cell 11 ("Compare seeds") for a single
`(features, checkpoint)` pair, but draws **one composite figure** per adapter:

```
+------------+  +----+ +----+ +----+   <- X/Y/Z component traces (100 seeds)
|            |  +----+ +----+ +----+
|  3D EEF    |
|  traj      |  +----+ +----+ +----+   <- x/y/z mean ± 1 std spread
|            |  +----+ +----+ +----+
+------------+
```

`analyze_all_checkpoints` drives it across every checkpoint for one observation,
encoding backbone features just once.

In [ ]:
# Cell 3: run_seed_sweep — reproduces v1 Cell 11 ("Compare seeds") for one
# (features, checkpoint) pair. Sweeps n_seeds noise seeds through the 4-step denoiser,
# decodes each to an EEF trajectory, prints per-component spread stats, then draws ONE
# composite figure: a large 3D EEF-trajectory panel (left), the X/Y/Z component traces
# (top-right), and the x/y/z mean+-std spread (bottom-right). Returns a stats dict for
# the cross-checkpoint summary.
def run_seed_sweep(lab, features, label, n_seeds=N_SEEDS, eef_key=EEF_KEY, show_plots=True):
    viz_seeds = TrajectoryVisualizer()
    for seed in range(n_seeds):
        r = lab.denoise(features, seed=seed)
        d = lab.decode_raw_actions(r.action_pred, features.states)
        viz_seeds.add_trajectory(d, f"seed={seed}", eef_key=eef_key)

    # Stack all trajectories: shape (n_seeds, T+1, 3)
    all_pos = np.stack([t["positions"] for t in viz_seeds.trajectories])
    mean = all_pos.mean(axis=0)   # (T+1, 3)
    var = all_pos.var(axis=0)     # (T+1, 3)
    rng = all_pos.max(axis=0) - all_pos.min(axis=0)  # (T+1, 3)

    comp = ["x", "y", "z"]
    print("Across all timesteps:")
    for i, l in enumerate(comp):
        print(f"  {l}: mean={mean[:, i].mean():.4f}, \tvar={var[:, i].mean():.4f}, \tstd={np.sqrt(var[:, i]).mean():.4f}, \trange={rng[:, i].mean():.4f}")
    print("Across final timestep only:")
    for i, l in enumerate(comp):
        print(f"  {l}: mean={mean[-1, i]:.4f}, \tvar={var[-1, i]:.4f}, \tstd={np.sqrt(var[-1, i]):.4f}, \trange={rng[-1, i]:.4f}")

    if show_plots:
        # ONE composite figure per adapter: 3D (left, full height) + 2x3 grid (right).
        comp_up = ["X", "Y", "Z"]
        fig = plt.figure(figsize=(20, 8), constrained_layout=True)
        gs = fig.add_gridspec(2, 5)

        # Left: 3D EEF trajectory — one line per seed, shared start (^) and mean end (square).
        ax3d = fig.add_subplot(gs[:, 0:2], projection="3d")
        for s in range(all_pos.shape[0]):
            p = all_pos[s]
            ax3d.plot(p[:, 0], p[:, 1], p[:, 2], "-", color="C0", linewidth=0.6, alpha=0.4)
        ax3d.scatter(*all_pos[0, 0], marker="^", s=60, color="k", zorder=5)
        ax3d.scatter(*mean[-1], marker="s", s=60, color="r", zorder=5)
        # Equal aspect via bounding box (mirrors TrajectoryVisualizer.plot_eef_3d).
        flat = all_pos.reshape(-1, 3)
        bmin, bmax = flat.min(axis=0), flat.max(axis=0)
        mid = (bmin + bmax) / 2
        half = (bmax - bmin).max() / 2
        if half < 1e-8:
            half = 0.01
        ax3d.set_xlim(mid[0] - half, mid[0] + half)
        ax3d.set_ylim(mid[1] - half, mid[1] + half)
        ax3d.set_zlim(mid[2] - half, mid[2] + half)
        ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")
        ax3d.set_title(f"3D EEF trajectory ({n_seeds} seeds)", fontsize=10)

        # Top-right: raw X/Y/Z component traces (one line per seed).
        for i in range(3):
            axc = fig.add_subplot(gs[0, 2 + i])
            axc.plot(all_pos[:, :, i].T, color="C0", linewidth=0.5, alpha=0.25)
            axc.set_title(f"EEF {comp_up[i]} ({n_seeds} seeds)", fontsize=9)
            axc.set_xlabel("timestep", fontsize=8)
            axc.grid(True, alpha=0.3)

        # Bottom-right: x/y/z mean +- 1 std spread.
        for i in range(3):
            axs = fig.add_subplot(gs[1, 2 + i])
            std_i = np.sqrt(var[:, i])
            axs.plot(mean[:, i], color="C1", label="mean")
            axs.fill_between(range(mean.shape[0]), mean[:, i] - std_i, mean[:, i] + std_i,
                             alpha=0.3, color="C1", label="±1 std")
            axs.set_title(f"{comp[i]} mean±std", fontsize=9)
            axs.set_xlabel("timestep", fontsize=8)
            axs.grid(True, alpha=0.3)
            if i == 0:
                axs.legend(fontsize=7)

        fig.suptitle(f"Seed spread — {label}", fontsize=13)
        plt.show()

    return {
        "label": label,
        "mean": mean,
        "var": var,
        "range": rng,
        "final_std": np.sqrt(var[-1]),          # (3,) std at final timestep
        "mean_std": np.sqrt(var).mean(axis=0),  # (3,) std averaged over timesteps
    }

In [ ]:
# Cell 4: analyze_all_checkpoints — encode one observation, then run the seed sweep
# for every checkpoint in CHECKPOINTS. Features are encoded ONCE (the backbone is
# LoRA-free); each checkpoint's adapter is loaded into the already-injected DiT, fully
# overwriting the previous checkpoint's A/B matrices.
def analyze_all_checkpoints(obs_path, n_seeds=N_SEEDS, show_obs=True):
    obs = DenoisingLab.load_observation(obs_path)
    print("Observation:", obs_path)
    print("Task:", obs["annotation.human.action.task_description"])
    if show_obs:
        DenoisingLab.plot_camera_views(obs, figsize=(14, 4))
        plt.suptitle(f"Camera views — {os.path.basename(obs_path)}", fontsize=12)
        plt.show()

    features = lab.encode_features_from_sim_obs(obs)  # EXPENSIVE — once per obs
    print(f"Backbone features: {features.backbone_features.shape}")

    results = {}
    for ckpt_rel in CHECKPOINTS:
        ckpt = os.path.join(REPO_ROOT, ckpt_rel)
        label = ckpt_rel.replace("grpo_data/", "")
        print("\n" + "=" * 90)
        print(f"CHECKPOINT: {label}")
        print("=" * 90)
        load_lora_checkpoint(lab.model, ckpt)  # overwrites the previous adapter in place
        results[label] = run_seed_sweep(lab, features, label, n_seeds=n_seeds)
    return features, results

In [ ]:
# Cell 5: Run the full comparison for OBS_PATH.
# Re-run with a different path for any other observation, e.g.:
#   _features, results = analyze_all_checkpoints("/path/to/other.npz")
_features, results = analyze_all_checkpoints(OBS_PATH)

## Cross-checkpoint summary

Final-timestep EEF-position std across the seeds, per checkpoint. Higher std = more
action diversity driven by the initial noise; lower std = the checkpoint has become
more "decisive" / peaked for this observation.

In [ ]:
# Cell 6: Cross-checkpoint summary — final-timestep EEF position std across seeds.
labels_short = list(results.keys())
final_std = np.array([results[k]["final_std"] for k in labels_short])  # (n_ckpt, 3)

print(f"{'checkpoint':52s} {'std_x':>8s} {'std_y':>8s} {'std_z':>8s} {'mean':>9s}")
for k, fs in zip(labels_short, final_std):
    print(f"{k:52s} {fs[0]:8.4f} {fs[1]:8.4f} {fs[2]:8.4f} {fs.mean():9.4f}")

x = np.arange(len(labels_short))
w = 0.25
fig, ax = plt.subplots(figsize=(15, 6))
for i, l in enumerate(["x", "y", "z"]):
    ax.bar(x + (i - 1) * w, final_std[:, i], w, label=f"std {l}")
ax.set_xticks(x)
ax.set_xticklabels(labels_short, rotation=70, ha="right", fontsize=7)
ax.set_ylabel("final-timestep EEF std across seeds")
ax.set_title(f"Action diversity by checkpoint — {os.path.basename(OBS_PATH)}")
ax.legend()
plt.tight_layout()
plt.show()